# Setup: `v2_patch` Table for Assign Labels Benchmark

## Schema Description

This notebook documents the **existing** `v2_patch` table used for the
**Assign Labels** benchmark. The table already contains ~800 M rows and must
**not** be dropped, truncated, or otherwise destructively modified.

The schema mirrors the `patch` table defined in `db_technical_design.md §1.3`:

| Column       | Type              | Description                                         |
|--------------|-------------------|-----------------------------------------------------|
| id           | INTEGER PK        | Sequential primary key                              |
| patch_uid    | INT NOT NULL      | Unique patch identifier                             |
| gt_label     | INT (nullable)    | Ground truth label — **the column being updated**   |
| event_ts     | TIMESTAMPTZ       | Timestamp when ground truth was created/last updated|
| image_id     | INT (nullable)    | FK-like reference to source image                   |
| working_mag  | FLOAT (nullable)  | Working magnification level                         |

**Table type**: `UNLOGGED` (avoids WAL overhead; existing benchmark convention).

**Index**: `PRIMARY KEY` B-tree index on `id` — used by the UPDATE WHERE clause.

**Row count**: ~800,000,200 rows (verified at setup time).

## Benchmark Strategy

The **Assign Labels** benchmark (§2.2 / §2.6 in the design doc) measures
the wall-clock time to upsert `gt_label` for 1,000 randomly-selected patches
inside this billion-row table. Because the table is live, the benchmark:

1. **Saves** the original `gt_label` values for the 1,000 chosen rows.
2. **Times** a single-statement `UPDATE … WHERE id = ANY(%s)` setting a new
   label for all 1,000 patches.
3. **Restores** the original `gt_label` values after the timed section.

## Connection
Reads credentials from environment variables `DB_HOST`, `DB_NAME`, `DB_USER`,
`DB_PASSWORD`; falls back to prototyping defaults if not set.

In [ ]:
import os
import psycopg2

# ---------------------------------------------------------------------------
# Connection parameters — read from env vars, fall back to prototyping defaults
# ---------------------------------------------------------------------------
DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')

DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

TABLE = 'v2_patch'

conn = psycopg2.connect(DSN)
cur  = conn.cursor()

# Report PG version
cur.execute('SELECT version();')
pg_ver = cur.fetchone()[0].split(',')[0]
print(f'Connected to {pg_ver}')

# ---------------------------------------------------------------------------
# Verify the table exists and report statistics
# ---------------------------------------------------------------------------
cur.execute(f"SELECT COUNT(*) FROM {TABLE};")
row_count = cur.fetchone()[0]
print(f'Table {TABLE!r} row count: {row_count:,}')

cur.execute(f"SELECT MIN(id), MAX(id) FROM {TABLE};")
min_id, max_id = cur.fetchone()
print(f'ID range: {min_id:,} – {max_id:,}')

# Report index info
cur.execute("""
    SELECT indexname, indexdef
    FROM pg_indexes
    WHERE tablename = %s;
""", (TABLE,))
print('\nIndexes:')
for row in cur.fetchall():
    print(f'  {row[0]}: {row[1]}')

# Report gt_label distribution
cur.execute(f"SELECT gt_label, COUNT(*) FROM {TABLE} GROUP BY gt_label ORDER BY gt_label;")
print('\ngt_label distribution:')
for row in cur.fetchall():
    print(f'  gt_label={row[0]}: {row[1]:,} rows')

# Report table persistence type
cur.execute("""
    SELECT relname, relpersistence
    FROM pg_class WHERE relname = %s;
""", (TABLE,))
row = cur.fetchone()
persistence = 'UNLOGGED' if row[1] == 'u' else 'PERMANENT'
print(f'\nTable persistence: {persistence}')

print('\nSetup verification complete — table is ready for benchmarking.')
conn.close()

In [ ]:
# ---------------------------------------------------------------------------
# TEARDOWN NOTE
# ---------------------------------------------------------------------------
# This benchmark operates on the EXISTING v2_patch table.
# There is NO destructive teardown (no DROP/TRUNCATE).
#
# The benchmark notebook itself saves the original gt_label values for the
# 1,000 sampled rows and RESTORES them after the timed section, so the table
# is left in exactly the state it was found in.
#
# If a run fails mid-way and you need to manually restore, you would need
# to re-run the restore block in the benchmark notebook.
print('No destructive teardown for v2_patch — the benchmark restores its own changes.')